# Aegis Prompt Injection Classifier

**ModelHub**: MH-2026-031  
**Task**: binary_classification (injection=1, safe=0)  
**Input**: text (prompt string)  
**Output**: injection_probability (float 0-1)  
**Dataset format**: JSONL `{"text": "...", "label": 0|1}`

## GPU 需求
- Kaggle T4/P100（每週 30 hr 免費配額）
- 預計訓練時間：~2 hr（DistilBERT fine-tune）

## 產出
- `model/` — HuggingFace tokenizer + model weights
- `metrics.json` — accuracy, f1, auc
- 上傳 Kaggle Dataset: `boardgamegroup/aegis-pi-model-v1`


In [ ]:
# Setup
import json
import os
from pathlib import Path

DATASET_PATH = Path('/kaggle/input/aegis-prompt-injection-dataset')
OUTPUT_PATH = Path('/kaggle/working/model')
OUTPUT_PATH.mkdir(exist_ok=True)

MODELHUB_REQ_NO = 'MH-2026-031'
MODELHUB_API_URL = os.environ.get('MODELHUB_API_URL', '')
MODELHUB_API_KEY = os.environ.get('MODELHUB_API_KEY', '')


In [ ]:
# Load dataset
data_file = DATASET_PATH / 'prompt_injection_dataset.jsonl'
rows = [json.loads(l) for l in data_file.read_text().splitlines() if l.strip()]
texts = [r['text'] for r in rows]
labels = [r['label'] for r in rows]
print(f'Samples: {len(texts)}, Positive: {sum(labels)}, Negative: {len(labels)-sum(labels)}')


In [ ]:
# Train DistilBERT binary classifier
# TODO: 等 Kaggle dataset 上傳後補完整訓練程式碼
# 參考架構：
#   from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, Trainer, TrainingArguments
#   tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
#   model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
#   ... TrainingArguments + Trainer.train() ...
print('Training placeholder — fill in after dataset upload')


In [ ]:
# Report training result to ModelHub
import urllib.request

metrics = {'accuracy': 0.0, 'f1': 0.0}  # replace with real metrics

if MODELHUB_API_URL and MODELHUB_API_KEY:
    payload = json.dumps({
        'accuracy': metrics['accuracy'],
        'f1_score': metrics['f1'],
        'model_path': str(OUTPUT_PATH),
        'notes': 'Kaggle T4 training',
    }).encode()
    req = urllib.request.Request(
        f'{MODELHUB_API_URL}/api/submissions/{MODELHUB_REQ_NO}/training-result',
        data=payload,
        headers={'x-api-key': MODELHUB_API_KEY, 'Content-Type': 'application/json'},
        method='PATCH',
    )
    with urllib.request.urlopen(req, timeout=10) as r:
        print('ModelHub updated:', r.status)
else:
    print('MODELHUB_API_URL/KEY not set, skip reporting')
